# Dataset Pipeline - Local Processing

**Purpose**: Process vulnerability datasets locally from `raw/` directory and export standardized JSONL files to `datasets/` folder.

**Prerequisites**: Run `00_setup.ipynb` first

**Output**: Three splits (train/val/test) with identical structure:

```json
{
  "lines": [normalized_code_lines],
  "raw_lines": [original_code_lines],
  "label": [0/1_vulnerability_labels],
  "type": [statement_types],
  "cwe_id": "CWE-XXX",
  "dataset_source": "source_name"
}
```


## Section 1: Imports & Configuration


In [60]:
import os
import json
import re
import ast
import textwrap
import difflib
import shutil
import random
import warnings
from collections import Counter
from pathlib import Path

# Suppress SyntaxWarnings from raw data regex patterns
warnings.filterwarnings('ignore', category=SyntaxWarning)

In [61]:
# Configure paths (works in both Colab and local)
try:
    from google.colab import drive
    # Running in Colab - use Google Drive
    drive.mount('/content/drive')
    BASE_DIR = Path("/content/drive/MyDrive/CSI_Project")
    DATASETS_DIR = BASE_DIR / "datasets"
    RAW_DIR = DATASETS_DIR / "raw"  # In Colab, raw data is inside datasets folder
    IS_COLAB = True
    print("✓ Running in Colab - using Google Drive paths")
except ImportError:
    # Running locally
    BASE_DIR = Path("/Users/anas/Projects/code-security-identifier")
    DATASETS_DIR = BASE_DIR / "datasets"
    RAW_DIR = BASE_DIR / "raw"  # Locally, raw data is at project root level
    IS_COLAB = False
    print("✓ Running locally - using local paths")

# Create directories
DATASETS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(42)

print(f"✓ Base directory: {BASE_DIR}")
print(f"✓ Raw data directory: {RAW_DIR}")
print(f"✓ Output directory: {DATASETS_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Running in Colab - using Google Drive paths
✓ Base directory: /content/drive/MyDrive/CSI_Project
✓ Raw data directory: /content/drive/MyDrive/CSI_Project/datasets/raw
✓ Output directory: /content/drive/MyDrive/CSI_Project/datasets


## Section 2: Helper Functions


In [62]:
# Statement type mapping for AST analysis
STMT_TYPES = {
    ast.Import: "Import",
    ast.ImportFrom: "Import From",
    ast.Assign: "Assign",
    ast.AugAssign: "Augmented assign",
    ast.AnnAssign: "Assign",
    ast.Return: "Return",
    ast.If: "Condition",
    ast.For: "For/While",
    ast.While: "For/While",
    ast.With: "Expression",
    ast.Assert: "Assert",
    ast.Expr: "Expression",
    ast.FunctionDef: "Function declaration",
    ast.AsyncFunctionDef: "Function declaration",
    ast.ClassDef: "Expression",
    ast.Try: "Expression",
    ast.Raise: "Expression",
    ast.Pass: "Expression",
    ast.Break: "Expression",
    ast.Continue: "Expression",
}


def split_into_statements(code_str):
    """
    Parse code into statements with types.
    Returns: (raw_lines, normalized_lines, types) or None if unparseable.
    """
    if not code_str or not code_str.strip():
        return None

    # Get statement types via AST
    line_types = {}
    try:
        tree = ast.parse(textwrap.dedent(code_str))
        for node in ast.walk(tree):
            if hasattr(node, "lineno") and isinstance(node, ast.stmt):
                line_types[node.lineno] = STMT_TYPES.get(type(node), "Expression")
    except SyntaxError:
        # Try wrapping in function
        try:
            wrapped = "def __w__():\n" + textwrap.indent(
                textwrap.dedent(code_str), "    "
            )
            tree = ast.parse(wrapped)
            for node in ast.walk(tree):
                if hasattr(node, "lineno") and isinstance(node, ast.stmt):
                    line_types[node.lineno - 1] = STMT_TYPES.get(
                        type(node), "Expression"
                    )
        except SyntaxError:
            pass

    dedented = textwrap.dedent(code_str)
    all_lines = dedented.split("\n")

    raw, norm, types = [], [], []
    for i, line in enumerate(all_lines):
        if not line.strip():
            continue
        raw.append(line)
        norm.append(" ".join(line.split()))

        s = line.strip()
        if s.startswith('"""') or s.startswith("'''"):
            types.append("Docstring")
        else:
            types.append(line_types.get(i + 1, "Expression"))

    return (raw, norm, types) if raw else None


def label_vulnerable_lines(raw_lines, bad_texts):
    """
    Label lines as vulnerable (1) or safe (0) based on bad code patterns.
    """
    labels = [0] * len(raw_lines)
    if not bad_texts:
        return labels

    for i, line in enumerate(raw_lines):
        lc = line.strip()
        if not lc:
            continue
        for bt in bad_texts:
            bc = bt.strip()
            if not bc:
                continue
            if bc in lc or lc in bc or " ".join(bc.split()) == " ".join(lc.split()):
                labels[i] = 1
                break
    return labels


def write_jsonl(path, records):
    """
    Write list of records to JSONL file.
    """
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")


def read_jsonl(path):
    """
    Read JSONL file into list of records.
    """
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return [json.loads(line) for line in f]


print("✓ Helper functions loaded")

✓ Helper functions loaded


## Section 3: Process Local Vudenc Dataset


In [63]:
# CWE mappings
CWE = {
    "command_injection": "CWE-077",
    "open_redirect": "CWE-601",
    "path_disclosure": "CWE-022",
    "remote_code_execution": "CWE-094",
    "sql": "CWE-089",
    "xsrf": "CWE-352",
    "xss": "CWE-079",
}

vudenc_records = []
skipped = 0

print("Processing local Vudenc dataset...")
for vuln_name, cwe_id in CWE.items():
    path = RAW_DIR / f"plain_{vuln_name}"
    if not path.exists():
        print(f"  ⊘ {vuln_name}: file not found")
        continue

    with open(path) as f:
        data = json.load(f)

    count = 0
    for repo, commits in data.items():
        for sha, info in commits.items():
            for fpath, fdata in info.get("files", {}).items():
                source = fdata.get("sourceWithComments", "") or fdata.get("source", "")
                changes = fdata.get("changes", [])

                bads = []
                for ch in changes:
                    if isinstance(ch, dict):
                        bads.extend(ch.get("badparts", []))

                if not source or not bads:
                    continue

                result = split_into_statements(source)
                if result is None:
                    skipped += 1
                    continue

                raw, norm, types = result
                labels = label_vulnerable_lines(raw, bads)

                if sum(labels) == 0:
                    skipped += 1
                    continue

                vudenc_records.append(
                    {
                        "lines": norm,
                        "raw_lines": raw,
                        "label": labels,
                        "type": types,
                        "cwe_id": cwe_id,
                        "dataset_source": "vudenc",
                    }
                )
                count += 1

    print(f"  ✓ {vuln_name} ({cwe_id}): {count:,} functions")

print(f"✓ Local Vudenc: {len(vudenc_records):,} functions ({skipped:,} skipped)")

Processing local Vudenc dataset...
  ✓ command_injection (CWE-077): 197 functions
  ✓ open_redirect (CWE-601): 182 functions
  ✓ path_disclosure (CWE-022): 232 functions
  ✓ remote_code_execution (CWE-094): 131 functions
  ✓ sql (CWE-089): 656 functions
  ✓ xsrf (CWE-352): 296 functions
  ✓ xss (CWE-079): 81 functions
✓ Local Vudenc: 1,775 functions (1 skipped)


## Section 4: Process Function-Level Dataset


In [64]:
func_records = []
skipped = 0

path = RAW_DIR / "function_level_dataset.out"
print("Processing function-level dataset...")

if not path.exists():
    print(f"  ⊘ function_level_dataset.out not found")
else:
    with open(path) as f:
        records = [json.loads(l) for l in f]

    for rec in records:
        if rec.get("programming_language") != "Python":
            continue

        before = rec.get("code_before", "")
        after = rec.get("code_after", "")

        if not before.strip():
            continue

        result = split_into_statements(before)
        if result is None:
            skipped += 1
            continue

        raw, norm, types = result

        # Diff before/after to find changed lines
        if after.strip():
            removed = set()
            for d in difflib.unified_diff(
                before.split("\n"), after.split("\n"), lineterm=""
            ):
                if d.startswith("-") and not d.startswith("---"):
                    removed.add(d[1:].strip())

            labels = [0] * len(raw)
            for i, line in enumerate(raw):
                if line.strip() in removed:
                    labels[i] = 1
        else:
            labels = [1] * len(raw)

        # Extract CWE from description - try multiple extraction strategies
        cwe_id = "unknown"
        
        # Strategy 1: Direct CWE-XXX format
        gpt = rec.get("gpt_answer", "") + " " + rec.get("description", "")
        m = re.search(r"CWE-\d+", gpt)
        if m:
            cwe_id = m.group(0)
        else:
            # Strategy 2: Map description keywords to known CWEs
            desc_lower = gpt.lower()
            cwe_keywords = {
                "sql injection": "CWE-089",
                "xss": "CWE-079",
                "cross-site": "CWE-079",
                "open redirect": "CWE-601",
                "path traversal": "CWE-022",
                "command injection": "CWE-077",
                "remote code": "CWE-094",
                "csrf": "CWE-352",
                "xsrf": "CWE-352",
            }
            for keyword, cwe in cwe_keywords.items():
                if keyword in desc_lower:
                    cwe_id = cwe
                    break

        func_records.append(
            {
                "lines": norm,
                "raw_lines": raw,
                "label": labels,
                "type": types,
                "cwe_id": cwe_id,
                "dataset_source": "funclevel",
            }
        )

        # Also add the fixed (safe) version
        if after.strip():
            r2 = split_into_statements(after)
            if r2:
                raw2, norm2, types2 = r2
                func_records.append(
                    {
                        "lines": norm2,
                        "raw_lines": raw2,
                        "label": [0] * len(raw2),
                        "type": types2,
                        "cwe_id": cwe_id,
                        "dataset_source": "funclevel",
                    }
                )

print(f"✓ Function-level: {len(func_records):,} functions ({skipped:,} skipped)")

Processing function-level dataset...
✓ Function-level: 3,043 functions (0 skipped)


## Section 5: Process Security Eval Dataset


In [65]:
sec_records = []

path = RAW_DIR / "dataset.jsonl"
print("Processing security eval dataset...")

if not path.exists():
    print(f"  ⊘ dataset.jsonl not found")
else:
    with open(path) as f:
        for line in f:
            rec = json.loads(line)
            cwe_id = rec["ID"].split("_")[0]

            result = split_into_statements(rec["Insecure_code"])
            if result is None:
                continue

            raw, norm, types = result
            sec_records.append(
                {
                    "lines": norm,
                    "raw_lines": raw,
                    "label": [1] * len(raw),
                    "type": types,
                    "cwe_id": cwe_id,
                    "dataset_source": "securityeval",
                }
            )

    cwe_count = len(set(r["cwe_id"] for r in sec_records))
    print(
        f"✓ Security Eval: {len(sec_records):,} functions, {cwe_count} unique CWE types"
    )

Processing security eval dataset...
✓ Security Eval: 121 functions, 69 unique CWE types


## Section 6: Combine & Split into Train/Val/Test


In [66]:
print("Combining all datasets...")

# Combine all data
train_all = vudenc_records + func_records + sec_records

print(f"  Total records (before dedup): {len(train_all):,}")

# Deduplicate based on code content + labels + CWE
import hashlib

seen = {}
deduped = []

for rec in train_all:
    # Create signature from code content, labels, and CWE (not dataset source)
    payload = {
        "lines": rec.get("lines", []),
        "label": rec.get("label", []),
        "cwe_id": rec.get("cwe_id", "unknown"),
    }
    sig = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
    
    if sig not in seen:
        seen[sig] = True
        deduped.append(rec)

train_all = deduped
print(f"  Total records (after dedup): {len(train_all):,}")

# Shuffle and split into train/val
random.shuffle(train_all)
val_size = max(1, len(train_all) // 10)  # 10% validation
final_val = train_all[:val_size]
final_train = train_all[val_size:]

print(f"  Train: {len(final_train):,}")
print(f"  Val:   {len(final_val):,}")

# Save to datasets folder
print("\nSaving datasets...")
for name, data in [("FINAL_train", final_train), ("FINAL_val", final_val)]:
    path = DATASETS_DIR / f"{name}.jsonl"
    write_jsonl(path, data)

    n = len(data)
    stmts = sum(len(r["label"]) for r in data)
    vuln = sum(sum(r["label"]) for r in data)
    pct = vuln / stmts * 100 if stmts > 0 else 0
    print(
        f"  ✓ {name}: {n:,} functions, {stmts:,} statements ({vuln:,} vulnerable, {pct:.1f}%)"
    )

print(f"\n✓ Datasets ready at: {DATASETS_DIR}")

Combining all datasets...
  Total records (before dedup): 4,939
  Total records (after dedup): 4,085
  Train: 3,677
  Val:   408

Saving datasets...
  ✓ FINAL_train: 3,677 functions, 257,310 statements (22,339 vulnerable, 8.7%)
  ✓ FINAL_val: 408 functions, 31,281 statements (2,842 vulnerable, 9.1%)

✓ Datasets ready at: /content/drive/MyDrive/CSI_Project/datasets


## Section 8: Data preprocessing: Convert & verify outputs

Run these checks at the end to ensure all converted JSONL outputs are structurally valid and ready for downstream training/validation notebooks.

In [67]:
import json
from collections import Counter
from pathlib import Path

REQUIRED_FIELDS = ["lines", "raw_lines", "label", "type", "cwe_id", "dataset_source"]

# Export intermediate converted datasets before verification
intermediate_exports = [
    ("local_vudenc_converted.jsonl", vudenc_records),
    ("local_funclevel_converted.jsonl", func_records),
    ("securityeval_eval.jsonl", sec_records),
]

print("Exporting converted intermediate datasets...")
for fname, records in intermediate_exports:
    out_path = DATASETS_DIR / fname
    write_jsonl(out_path, records)
    print(f"  ✓ {fname}: {len(records):,} records")
print()


def verify_record(record, idx):
    issues = []

    for field in REQUIRED_FIELDS:
        if field not in record:
            issues.append(f"record[{idx}] missing '{field}'")

    n_labels = len(record.get("label", []))
    n_lines = len(record.get("lines", []))
    n_raw = len(record.get("raw_lines", []))
    n_types = len(record.get("type", []))

    if n_labels == 0:
        issues.append(f"record[{idx}] empty label list")
    if n_labels != n_lines:
        issues.append(f"record[{idx}] label({n_labels}) != lines({n_lines})")
    if n_labels != n_raw:
        issues.append(f"record[{idx}] label({n_labels}) != raw_lines({n_raw})")
    if n_labels != n_types:
        issues.append(f"record[{idx}] label({n_labels}) != type({n_types})")

    bad_labels = [x for x in record.get("label", []) if x not in (0, 1)]
    if bad_labels:
        issues.append(f"record[{idx}] has non-binary labels")

    return issues


def verify_jsonl_file(path: Path):
    result = {
        "path": str(path),
        "exists": path.exists(),
        "records": 0,
        "statements": 0,
        "vulnerable_statements": 0,
        "vulnerable_functions": 0,
        "issues": [],
        "dataset_sources": {},
        "top_cwe": [],
    }

    if not path.exists():
        result["issues"].append("file not found")
        return result

    source_counter = Counter()
    cwe_counter = Counter()

    with open(path, "r") as f:
        for idx, line in enumerate(f):
            line = line.strip()
            if not line:
                result["issues"].append(f"line {idx + 1}: empty line")
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                result["issues"].append(f"line {idx + 1}: invalid JSON")
                continue

            result["records"] += 1
            result["statements"] += len(record.get("label", []))
            result["vulnerable_statements"] += sum(record.get("label", []))
            result["vulnerable_functions"] += int(sum(record.get("label", [])) > 0)

            source_counter[record.get("dataset_source", "unknown")] += 1
            cwe_counter[record.get("cwe_id", "unknown")] += 1

            result["issues"].extend(verify_record(record, idx))

    result["dataset_sources"] = dict(source_counter)
    result["top_cwe"] = cwe_counter.most_common(10)
    return result


files_to_verify = [
    DATASETS_DIR / "local_vudenc_converted.jsonl",
    DATASETS_DIR / "local_funclevel_converted.jsonl",
    DATASETS_DIR / "securityeval_eval.jsonl",
    DATASETS_DIR / "FINAL_train.jsonl",
    DATASETS_DIR / "FINAL_val.jsonl",
]

report = {"files": {}, "summary": {}}

print("Verifying converted/preprocessed outputs...\n")
for path in files_to_verify:
    file_report = verify_jsonl_file(path)
    report["files"][path.name] = file_report

    vuln_ratio = (
        file_report["vulnerable_statements"] / file_report["statements"] * 100
        if file_report["statements"] > 0
        else 0
    )

    print(f"{path.name}")
    print(f"  records: {file_report['records']:,}")
    print(f"  statements: {file_report['statements']:,}")
    print(
        f"  vulnerable statements: {file_report['vulnerable_statements']:,} ({vuln_ratio:.1f}%)"
    )
    print(f"  issues: {len(file_report['issues'])}")

    if file_report["issues"]:
        for issue in file_report["issues"][:5]:
            print(f"    - {issue}")
        if len(file_report["issues"]) > 5:
            print(f"    - ... and {len(file_report['issues']) - 5} more")
    print()

all_issues = sum(len(r["issues"]) for r in report["files"].values())
all_records = sum(r["records"] for r in report["files"].values())
all_statements = sum(r["statements"] for r in report["files"].values())
all_vuln_statements = sum(r["vulnerable_statements"] for r in report["files"].values())

report["summary"] = {
    "total_files": len(files_to_verify),
    "total_records": all_records,
    "total_statements": all_statements,
    "total_vulnerable_statements": all_vuln_statements,
    "total_issues": all_issues,
}

report_path = DATASETS_DIR / "preprocessing_quality_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print("=" * 70)
print(f"Preprocessing quality report written to: {report_path}")
print(f"Total files checked: {len(files_to_verify)}")
print(f"Total issues found: {all_issues}")
print("=" * 70)

if all_issues == 0:
    print("PASS: Converted outputs are structurally valid.")
else:
    print("WARN: Review preprocessing_quality_report.json before training.")

Exporting converted intermediate datasets...
  ✓ local_vudenc_converted.jsonl: 1,775 records
  ✓ local_funclevel_converted.jsonl: 3,043 records
  ✓ securityeval_eval.jsonl: 121 records

Verifying converted/preprocessed outputs...

local_vudenc_converted.jsonl
  records: 1,775
  statements: 317,495
  vulnerable statements: 26,154 (8.2%)
  issues: 0

local_funclevel_converted.jsonl
  records: 3,043
  statements: 96,225
  vulnerable statements: 6,289 (6.5%)
  issues: 0

securityeval_eval.jsonl
  records: 121
  statements: 1,412
  vulnerable statements: 1,412 (100.0%)
  issues: 0

FINAL_train.jsonl
  records: 3,677
  statements: 257,310
  vulnerable statements: 22,339 (8.7%)
  issues: 0

FINAL_val.jsonl
  records: 408
  statements: 31,281
  vulnerable statements: 2,842 (9.1%)
  issues: 0

Preprocessing quality report written to: /content/drive/MyDrive/CSI_Project/datasets/preprocessing_quality_report.json
Total files checked: 5
Total issues found: 0
PASS: Converted outputs are structurally

In [68]:
# Final Summary
print("\n" + "=" * 70)
print("✓ DATA PIPELINE PROCESSING COMPLETE")
print("=" * 70)
print("\nOutput files created in: " + str(DATASETS_DIR))

for output_file in ["FINAL_train.jsonl", "FINAL_val.jsonl", "preprocessing_quality_report.json"]:
    output_path = DATASETS_DIR / output_file
    if output_path.exists():
        size_mb = output_path.stat().st_size / (1024 * 1024)
        print(f"  ✓ {output_file} ({size_mb:.2f}MB)")

print(f"\n✓ Raw data preserved at: {RAW_DIR}")
print("  Raw data files are NOT deleted - they remain for reference")
print("\nNext step: Run 02_validation_analysis.ipynb to validate data quality")
print("=" * 70)


✓ DATA PIPELINE PROCESSING COMPLETE

Output files created in: /content/drive/MyDrive/CSI_Project/datasets
  ✓ FINAL_train.jsonl (25.43MB)
  ✓ FINAL_val.jsonl (3.02MB)
  ✓ preprocessing_quality_report.json (0.00MB)

✓ Raw data preserved at: /content/drive/MyDrive/CSI_Project/datasets/raw
  Raw data files are NOT deleted - they remain for reference

Next step: Run 02_validation_analysis.ipynb to validate data quality
